# Lab 13: Put the assistant behind a web UI and share a link
**Module 7.1: Deployment patterns.**  **Time:** 25 minutes.

**Goal:** go from notebook code to something a colleague can use. We wrap the RAG assistant in a chat UI with
**Gradio** and get a public link. Then we map what a production deployment adds.

In **Colab** you get a temporary public `https://....gradio.live` link (valid for about a week, stops when the
notebook stops). **Locally** the app opens on `http://127.0.0.1:7860`.

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();
import os
lk.pip_install("gradio", module="gradio")
import gradio as gr
qc, _ = lk.build_policy_index()

## 1. The app's one function
Gradio calls this function for every chat message. Notice that the business logic is the same `rag_answer`.

In [ ]:
def respond(message, history):
    masked, _ = lk.mask_pii(message)
    r = lk.rag_answer(masked, qc)
    src = "\n\nSources: " + ", ".join(r["sources"]) if r["sources"] else ""
    return r["answer"] + src

print(respond("How many days of annual leave can I carry forward?", []))

## 2. Launch
Run the cell. In **Colab** it prints a `Running on public URL: https://....gradio.live` line and also shows the app
under the cell; in **VS Code** open `http://127.0.0.1:7860` in your browser.

**The cell keeps running while the app is live** (the run icon keeps spinning). That is expected. When you are
finished, press the cell's **stop** button (Colab: the square on the left of the cell; VS Code: the square
"Interrupt" button at the top). Stop it before you run any other cell.

Ask these three questions in the app, then share the link with your neighbour:
1. `What is the hotel limit in Pune?` (expected: INR 4,500 per night, because Pune is not a metro city in the policy)
2. `My email is ravi@gmail.com. How many sick days do I get?` (expected: 12 days; your e-mail was masked before the
   question reached the model)
3. `Who won the cricket match yesterday?` (expected: "I don't know based on the policies.")

In [ ]:
demo = gr.ChatInterface(respond, title="Meridian Policy Assistant",
                        description="Ask about leave, travel, IT, refunds and more. Answers cite the policy document.",
                        examples=["How many sick leave days do I get?", "What is the hotel limit in Pune?",
                                  "Can I work from another city?"])
if os.environ.get("LAB_NO_LAUNCH"):
    print("launch skipped (test mode)")
else:
    demo.launch(share=lk.IN_COLAB)

## 3. What production adds (discussion)
Your trainer will demo the same assistant running on a serverless edge platform with a real domain name.
Compare the two:

| Concern | This notebook | Production |
|---|---|---|
| Hosting | your Colab VM, temporary link | serverless or containers, autoscaling |
| Identity | anyone with the link | SSO, user identity passed to retrieval filters |
| Secrets | Colab secrets | secret manager, rotation |
| Rate limits | none | per user and per tenant, at an API gateway |
| Caching | none | cache frequent questions and embeddings |
| Index updates | rebuilt on start | pipeline triggered when documents change |
| Logs | printed | central logging and tracing (Lab 14) |

## Discussion (end of lab)
Which three rows would you insist on before letting 5,000 employees use it?